# Atendimento e Suporte para TI do MP

Esse é um projeto com a finalidade de usar um chatbot com RAG que responde com base em documentos reais da Instituição (ex.: manuais, PDFs).

* Retrieval-Augmented Generation (RAG) cuja técnica combina modelos de linguagem com mecanismos de recuperação de informações visando melhorar a geração de texto.

O LangChain será usado e tem vários componentes projetados para ajudar a criar aplicativos de perguntas e respostas e aplicativos RAG de forma mais geral.




# Instalação das bibliotecas

In [ ]:
!pip install -q langchain-core==0.3.1 langchain==0.3.1 langchain-community==0.3.1
!pip install -q langchain-text-splitters langchain-huggingface==0.1.1 langchain-groq
!pip install -q langchain-community==0.3.1
!pip install -q chromadb==0.5.20
!pip install -q sentence-transformers==3.1.0
!pip install -q torch torchvision

!pip install -q PyMuPDF python-docx
!pip install -q numpy==1.26.3
import langchain
langchain.verbose = False

ERROR: Cannot install langchain-core==0.3.1 and langchain==0.3.1 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.25 requires langsmith<0.4,>=0.1.17, but you have langsmith 0.10.9 which is incompatible.
langchain-community 0.3.1 requires langsmith<0.2.0,>=0.1.125, but you have langsmith 0.10.9 which is incompatible.
langgraph-prebuilt 1.1.0 requires langchain-core>=1.3.1, but you have langchain-core 0.3.86 which is incompatible.
langgraph-sdk 0.4.2 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.3.86 which is incompatible.
langgraph-sdk 0.4.2 requires websockets<16,>=14, but you have websockets 16.1.1 which is incompatible.
langgraph 1

# Importações e Carregamento da LLM

In [ ]:
import os
import getpass
from pathlib import Path

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyMuPDFLoader, Docx2txtLoader, TextLoader
from langchain.chains import create_history_aware_retriever, create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

# --- Configuração da LLM ---
os.environ["GROQ_API_KEY"] = getpass.getpass("Digite sua GROQ API Key: ")

id_model = "llama-3.3-70b-versatile"  # @param {type:"string"}
temperature = 0.7  # @param {type:"slider", min:0.1, max:1.5, step:0.1}

llm = ChatGroq(model=id_model, temperature=temperature, max_tokens=None, max_retries=2)
print("✅ LLM carregada!")

Digite sua GROQ API Key: ··········
✅ LLM carregada!


# Carregamento do Arquivo a ser lido

In [ ]:
from google.colab import files

uploaded = files.upload()
print(f"\n📄 Arquivos enviados: {list(uploaded.keys())}")

Saving MANUAL_DE_ATENDIMENTO_E_SUPORTE_DE_TI_MP.pdf to MANUAL_DE_ATENDIMENTO_E_SUPORTE_DE_TI_MP (4).pdf

📄 Arquivos enviados: ['MANUAL_DE_ATENDIMENTO_E_SUPORTE_DE_TI_MP (4).pdf']


In [ ]:
def carregar_documentos(pasta="/content"):
    """Carrega PDF, DOCX e TXT de uma pasta e retorna lista de textos."""
    docs_path = Path(pasta)
    arquivos = list(docs_path.glob("*"))
    documentos = []

    for arq in arquivos:
        ext = arq.suffix.lower()
        try:
            if ext == ".pdf":
                loader = PyMuPDFLoader(str(arq))
                pages = loader.load()
                documentos.append("\n".join([p.page_content for p in pages]))
            elif ext == ".docx":
                loader = Docx2txtLoader(str(arq))
                documentos.append(loader.load()[0].page_content)
            elif ext == ".txt":
                loader = TextLoader(str(arq), encoding="utf-8")
                documentos.append(loader.load()[0].page_content)
            print(f"  ✅ {arq.name}")
        except Exception as e:
            print(f"  ❌ {arq.name}: {e}")

    return documentos

def configurar_retriever(pasta="/content"):
    """Pipeline completa: carregar → dividir → embeddings → ChromaDB → retriever."""
    # 1. Carregar
    documentos = carregar_documentos(pasta)
    print(f"\n📚 {len(documentos)} documento(s) carregado(s)")

    # 2. Dividir em chunks
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = []
    for doc in documentos:
        chunks.extend(splitter.split_text(doc))
    print(f"✂️ {len(chunks)} chunks criados")

    # 3. Embeddings + ChromaDB
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
    vectorstore = Chroma.from_texts(chunks, embedding=embeddings, persist_directory="./chroma_db")
    print("💾 ChromaDB indexado e salvo em ./chroma_db")

    # 4. Retriever
    retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 3, "fetch_k": 4})
    return retriever

retriever = configurar_retriever()

  ✅ .config
  ✅ MANUAL_DE_ATENDIMENTO_E_SUPORTE_DE_TI_MP.pdf
  ✅ MANUAL_DE_ATENDIMENTO_E_SUPORTE_DE_TI_MP (1).pdf
  ✅ MANUAL_DE_ATENDIMENTO_E_SUPORTE_DE_TI_MP (4).pdf
  ✅ MANUAL_DE_ATENDIMENTO_E_SUPORTE_DE_TI_MP (2).pdf
  ✅ MANUAL_DE_ATENDIMENTO_E_SUPORTE_DE_TI_MP (3).pdf
  ✅ sample_data

📚 5 documento(s) carregado(s)
✂️ 105 chunks criados


/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


💾 ChromaDB indexado e salvo em ./chroma_db


# Definição do Contexto

In [ ]:
SYSTEM_PROMPT = """Você é um assistente virtual prestativo de TI do Ministério Público.
Use o contexto recuperado para responder à pergunta do usuário.
Se não souber a resposta, diga que não sabe com certeza.
Se for uma dúvida comum, sugira uma solução alternativa possível.
Mantenha a resposta concisa. Responda em português."""

CONTEXT_Q_PROMPT = "Given the chat history and the follow-up question, reformulate it as a standalone question. Do NOT answer, just reformulate if needed."

def configurar_rag(llm, retriever):
    # Reformulação da pergunta com base no histórico
    context_q_prompt = ChatPromptTemplate.from_messages([
        ("system", CONTEXT_Q_PROMPT),
        MessagesPlaceholder("chat_history"),
        ("human", "Question: {input}"),
    ])
    history_retriever = create_history_aware_retriever(llm, retriever, context_q_prompt)

    # Q&A chain
    qa_prompt = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_PROMPT),
        MessagesPlaceholder("chat_history"),
        ("human", "Pergunta: {input}\n\nContexto: {context}"),
    ])
    qa_chain = create_stuff_documents_chain(llm, qa_prompt)
    return create_retrieval_chain(history_retriever, qa_chain)

rag_chain = configurar_rag(llm, retriever)
print("✅ Chain RAG configurada!")

✅ Chain RAG configurada!


In [ ]:
def chat(rag_chain, pergunta, historico):
    """Envia pergunta à chain RAG, atualiza e retorna o histórico."""
    historico.append(HumanMessage(content=pergunta))
    resposta = rag_chain.invoke({"input": pergunta, "chat_history": historico})
    res = resposta["answer"]
    if "</think>" in res:
        res = res.split("</think>")[-1].strip()
    historico.append(AIMessage(content=res))
    return res, historico

# --- Teste ---
historico = []
res, historico = chat(rag_chain, "Como é a abertura de chamado?", historico)
print(f"🏛️ {res}\n")

# Pergunta de follow-up (usa o histórico)
res, historico = chat(rag_chain, "E pelo site, como faz?", historico)
print(f"🏛️ {res}")

🏛️ A abertura de chamado é o início do processo de atendimento e é responsabilidade do usuário solicitante. Para evitar atrasos, o usuário deve fornecer informações precisas, incluindo:

- Assunto claro
- Categoria (Rede, Software, Hardware)
- Número de Patrimônio do equipamento
- Localização exata (Bloco/Sala)
- Descrição detalhada do problema, incluindo:
  - O que estava fazendo quando o problema ocorreu
  - Qual erro apareceu (incluindo código de erro, se aplicável)
  - Quando o problema começou

O prazo para abertura do chamado é imediato após a detecção da falha.

🏛️ Para abrir um chamado pelo site, você deve acessar o Portal de Chamados (Help-Desk) e seguir os passos abaixo:

1. Acesse o site do Portal de Chamados.
2. Clique em "Abrir Chamado" ou "Novo Chamado".
3. Preencha o formulário com as informações necessárias, incluindo:
 * Assunto claro
 * Categoria (Rede, Software, Hardware)
 * Número de Patrimônio do equipamento
 * Localização exata (Bloco/Sala)
 * Descrição detalhada 